# Evered 2023 calibrated 9D Eq. (2) validation

This notebook intentionally does **not** import `neutral_yb` package models or optimizers. It rebuilds the closed-system Hamiltonian, the Methods Eq. (1) phase pulse, the light-shift/resonance calibration, and the process-fidelity check using only `numpy` and `scipy`.

Goal: check whether the Evered et al. Methods Eq. (1) pulse is a high-process-fidelity phase solution of a simplified Eq. (2)-style 9D ladder **after** applying the two-photon resonance calibration needed when the reported single-photon Rabi rates are unequal.


In [1]:
from __future__ import annotations

import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize_scalar

np.set_printoptions(precision=9, suppress=True)


## Paper parameters and basis

All Hamiltonians below are in units of the paper effective two-photon Rabi frequency `Omega`, with `Omega/2pi = 4.6 MHz`. The quantity `DETUNING_MHZ = 7800` is the intermediate-state detuning `Delta`, not the two-photon detuning `delta(t)`.


In [2]:
OMEGA_OVER_2PI_MHZ = 4.6
BLUE_RABI_MHZ = 237.0
RED_RABI_MHZ = 303.0
DETUNING_MHZ = 7800.0
BLOCKADE_MHZ = 450.0

blue_over_omega = BLUE_RABI_MHZ / OMEGA_OVER_2PI_MHZ
red_over_omega = RED_RABI_MHZ / OMEGA_OVER_2PI_MHZ
delta_over_omega = DETUNING_MHZ / OMEGA_OVER_2PI_MHZ
blockade_over_omega = BLOCKADE_MHZ / OMEGA_OVER_2PI_MHZ

effective_rabi_from_single_photon = blue_over_omega * red_over_omega / (2.0 * delta_over_omega)
balanced_rabi = np.sqrt(2.0 * delta_over_omega)

A_paper = 2.0 * np.pi * 0.1122
omega_paper = 1.0431
phi0_paper = -0.7318
delta0_paper = 0.0
omegaT_over_2pi_paper = 1.215
T_paper = 2.0 * np.pi * omegaT_over_2pi_paper

basis9 = ("01", "0e", "0r", "11", "W_e", "ee", "W_r", "E_er", "rr")

print("paper single-photon ratios:")
print("Omega_b/Omega =", blue_over_omega)
print("Omega_r/Omega =", red_over_omega)
print("Delta/Omega   =", delta_over_omega)
print("B/Omega       =", blockade_over_omega)
print("Omega_b Omega_r/(2 Delta Omega) =", effective_rabi_from_single_photon)
print("sqrt(2 Delta/Omega) balanced Rabi =", balanced_rabi)
print("paper pulse A/2pi, omega, phi0, OmegaT/2pi =", A_paper/(2*np.pi), omega_paper, phi0_paper, omegaT_over_2pi_paper)


paper single-photon ratios:
Omega_b/Omega = 51.52173913043479
Omega_r/Omega = 65.86956521739131
Delta/Omega   = 1695.6521739130435
B/Omega       = 97.82608695652175
Omega_b Omega_r/(2 Delta Omega) = 1.0007107023411375
sqrt(2 Delta/Omega) balanced Rabi = 58.23490660957641
paper pulse A/2pi, omega, phi0, OmegaT/2pi = 0.11219999999999998 1.0431 -0.7318 1.215


## What `paper_rabi_9d` means here

`paper_rabi_9d` is the calibrated single-intermediate-state version of the paper's Methods Eq. (2) model, lifted to the two-atom symmetric subspace needed for a global CZ check. The two-photon detuning zero is defined after the leading-order light-shift resonance calibration described below.

The one-atom Eq. (2) Hamiltonian is written in the `|1>, |e>, |r>` basis as

```text
H_3 = [[0,          Omega_b/2,       0],
       [Omega_b/2, -Delta,          Omega_r/2],
       [0,          Omega_r/2,      -delta(t)]]
```

The two-atom basis used here is

```text
|01>, |0e>, |0r>, |11>, |W_e>, |ee>, |W_r>, |E_er>, |rr>
```

where `|W_e>=(|1e>+|e1>)/sqrt(2)`, `|W_r>=(|1r>+|r1>)/sqrt(2)`, and `|E_er>=(|er>+|re>)/sqrt(2)`. The drift Hamiltonian contains the real 420-nm and 1,013-nm couplings with the paper-reported single-photon values

```text
Omega_b/Omega = 237/4.6 = 51.521739...
Omega_r/Omega = 303/4.6 = 65.869565...
Delta/Omega   = 7800/4.6 = 1695.652174...
B/Omega       = 450/4.6  = 97.826087...
```

The paper states the convention `Omega = Omega_b Omega_r / (2 Delta)`; with the reported values this gives `Omega_eff/Omega = 1.0007107`, close to the unit scale used by Eq. (1).

The control is represented in the Eq. (2) detuning gauge. The phase pulse

```text
phi(t) = A cos(omega t - phi0) + delta0 t
```

is converted to two-photon detuning as

```text
delta_waveform(t) = -d phi/dt = A omega sin(omega t - phi0) - delta0.
```

Because `Omega_b != Omega_r`, the bare single-intermediate-state ladder has a differential AC Stark shift. The minimal resonance calibration used here shifts the two-photon detuning zero by the leading-order amount

```text
delta_res = (Omega_r^2 - Omega_b^2) / (4 Delta).
```

Thus the propagated Hamiltonian is

```text
H_9(t) = H_drift + [delta_waveform(t) + delta_res] H_delta.
```

The fidelity is evaluated in the leading-order blue-light dressed computational basis. This models the Methods statement that turning on the 420-nm light adiabatically dresses `|1>` before the gate. The branch vectors used for the process overlap are

```text
|01_tilde> = normalize(|01> + eps |0e>)
|11_tilde> = normalize(|11> + sqrt(2) eps |W_e> + eps^2 |ee>)
eps = Omega_b / (2 Delta).
```

This is still a minimal closed-system validation model. It does not include the paper's full experimental ingredients: multiple intermediate states, Clebsch-Gordan factors, the adjacent `m_J` Rydberg state, AOM rise/fall, scattering, Rydberg decay, measured dephasing/noise, finite-temperature blockade variation, or the randomized-benchmark/SPAM extraction layer.



## From-scratch calibrated Hamiltonian builders


In [3]:
def add_quadrature_coupling(hx, hy, left, right, strength):
    hx[left, right] = strength
    hx[right, left] = strength
    hy[left, right] = -1j * strength
    hy[right, left] = 1j * strength


def two_photon_9d_detuning_gauge_matrices(lower_rabi, upper_rabi, detuning, blockade):
    drift = np.zeros((9, 9), dtype=np.complex128)
    drift[1, 1] = -detuning
    drift[4, 4] = -detuning
    drift[5, 5] = -2.0 * detuning
    drift[7, 7] = -detuning
    drift[8, 8] = blockade

    blue_x = np.zeros((9, 9), dtype=np.complex128)
    blue_y = np.zeros((9, 9), dtype=np.complex128)
    add_quadrature_coupling(blue_x, blue_y, 0, 1, 0.5)
    add_quadrature_coupling(blue_x, blue_y, 3, 4, 1.0 / np.sqrt(2.0))
    add_quadrature_coupling(blue_x, blue_y, 4, 5, 1.0 / np.sqrt(2.0))
    add_quadrature_coupling(blue_x, blue_y, 6, 7, 0.5)

    red_x = np.zeros((9, 9), dtype=np.complex128)
    red_y = np.zeros((9, 9), dtype=np.complex128)
    add_quadrature_coupling(red_x, red_y, 1, 2, 0.5)
    add_quadrature_coupling(red_x, red_y, 4, 6, 0.5)
    add_quadrature_coupling(red_x, red_y, 5, 7, 1.0 / np.sqrt(2.0))
    add_quadrature_coupling(red_x, red_y, 7, 8, 1.0 / np.sqrt(2.0))

    detuning_control = np.zeros((9, 9), dtype=np.complex128)
    detuning_control[2, 2] = -1.0
    detuning_control[6, 6] = -1.0
    detuning_control[7, 7] = -1.0
    detuning_control[8, 8] = -2.0

    drift = drift + lower_rabi * blue_x + upper_rabi * red_x
    return drift, detuning_control


def leading_order_resonance_shift(lower_rabi, upper_rabi, detuning):
    ground_light_shift = lower_rabi**2 / (4.0 * detuning)
    rydberg_light_shift = upper_rabi**2 / (4.0 * detuning)
    return rydberg_light_shift - ground_light_shift, ground_light_shift, rydberg_light_shift


def leading_order_blue_dressed_vectors(lower_rabi, detuning):
    eps = lower_rabi / (2.0 * detuning)
    v01 = np.zeros(9, dtype=np.complex128)
    v01[0] = 1.0
    v01[1] = eps
    v01 /= np.linalg.norm(v01)

    v11 = np.zeros(9, dtype=np.complex128)
    v11[3] = 1.0
    v11[4] = np.sqrt(2.0) * eps
    v11[5] = eps**2
    v11 /= np.linalg.norm(v11)
    return v01, v11, eps


def propagate_detuning_gauge(drift, h_delta, initial_state, T, nslots, A, omega, phi0, static_resonance_shift=0.0, delta0=0.0):
    dt = T / nslots
    state = initial_state.astype(np.complex128).copy()
    for k in range(nslots):
        t = (k + 0.5) * dt
        delta_t = A * omega * np.sin(omega * t - phi0) - delta0 + static_resonance_shift
        state = expm(-1j * dt * (drift + delta_t * h_delta)) @ state
    return state


def process_fidelity_from_branch_amplitudes(alpha, beta, theta):
    overlap = 1.0 + 2.0 * np.exp(-1j * theta) * alpha - np.exp(-2j * theta) * beta
    return float(abs(overlap) ** 2 / 16.0)


def optimize_theta_from_branch_amplitudes(alpha, beta):
    result = minimize_scalar(
        lambda th: -process_fidelity_from_branch_amplitudes(alpha, beta, float(th)),
        bounds=(0.0, 2.0 * np.pi),
        method="bounded",
        options={"xatol": 1e-11},
    )
    theta = float(np.mod(result.x, 2.0 * np.pi))
    return theta, process_fidelity_from_branch_amplitudes(alpha, beta, theta)


def branch_result(state, projector_01, projector_11):
    alpha = np.vdot(projector_01, state)
    beta = np.vdot(projector_11, state)
    theta, fpro = optimize_theta_from_branch_amplitudes(alpha, beta)
    leakage = float(max(0.0, 2.0 - (abs(alpha) ** 2 + abs(beta) ** 2)))
    return {
        "fidelity": fpro,
        "theta": theta,
        "leakage_proxy": leakage,
        "alpha": alpha,
        "beta": beta,
    }


def calibrated_paper_rabi_9d_check(nslots=800):
    drift, h_delta = two_photon_9d_detuning_gauge_matrices(
        blue_over_omega, red_over_omega, delta_over_omega, blockade_over_omega
    )
    delta_res, ground_shift, rydberg_shift = leading_order_resonance_shift(
        blue_over_omega, red_over_omega, delta_over_omega
    )
    dressed_01, dressed_11, eps = leading_order_blue_dressed_vectors(blue_over_omega, delta_over_omega)
    bare_01 = np.eye(9, dtype=np.complex128)[0]
    bare_11 = np.eye(9, dtype=np.complex128)[3]

    bare_state = propagate_detuning_gauge(
        drift,
        h_delta,
        bare_01 + bare_11,
        T_paper,
        nslots,
        A_paper,
        omega_paper,
        phi0_paper,
        static_resonance_shift=delta_res,
        delta0=delta0_paper,
    )
    dressed_state = propagate_detuning_gauge(
        drift,
        h_delta,
        dressed_01 + dressed_11,
        T_paper,
        nslots,
        A_paper,
        omega_paper,
        phi0_paper,
        static_resonance_shift=delta_res,
        delta0=delta0_paper,
    )

    return {
        "delta_res": delta_res,
        "ground_shift": ground_shift,
        "rydberg_shift": rydberg_shift,
        "eps": eps,
        "calibrated_bare": branch_result(bare_state, bare_01, bare_11),
        "calibrated_dressed": branch_result(dressed_state, dressed_01, dressed_11),
    }



## Calibrated paper-pulse check

This notebook retains two calibrated results. `calibrated_bare_paper_rabi_9d` uses the light-shift/resonance calibration but still prepares and projects the bare `|01>, |11>` branches. `calibrated_dressed_paper_rabi_9d` also prepares and projects the leading-order blue-light dressed branches described above.


In [4]:
result = calibrated_paper_rabi_9d_check(nslots=800)
print("calibrated paper_rabi_9d parameters in units of Omega:")
print(f"ground |1> light shift  = {result['ground_shift']:.12f}")
print(f"Rydberg |r> light shift = {result['rydberg_shift']:.12f}")
print(f"delta_res               = {result['delta_res']:.12f}")
print(f"blue dressing eps        = {result['eps']:.12f}")
print()
for label in ("calibrated_bare", "calibrated_dressed"):
    row = result[label]
    print(f"{label}_paper_rabi_9d")
    print(f"F_process     = {row['fidelity']:.12f}")
    print(f"theta         = {row['theta']:.9f}")
    print(f"leakage_proxy = {row['leakage_proxy']:.6e}")
    print(f"alpha         = {row['alpha']:.9f}")
    print(f"beta          = {row['beta']:.9f}")
    print()


calibrated paper_rabi_9d parameters in units of Omega:
ground |1> light shift  = 0.391367056856
Rydberg |r> light shift = 0.639694816054
delta_res               = 0.248327759197
blue dressing eps        = 0.015192307692

calibrated_bare_paper_rabi_9d
F_process     = 0.999550489372
theta         = 5.405766179
leakage_proxy = 8.942045e-04
alpha         = 0.631626566-0.774746687j
beta          = 0.182997336+0.983073356j

calibrated_dressed_paper_rabi_9d
F_process     = 0.999978311977
theta         = 5.405676388
leakage_proxy = 2.023543e-06
alpha         = 0.631998415-0.774969122j
beta          = 0.183181161+0.983078584j



## Time-discretization convergence

The table below checks that the retained calibrated result is not a time-slicing artifact.


In [5]:
nslot_grid = [80, 160, 320, 640, 800, 1000]
for nslots in nslot_grid:
    row = calibrated_paper_rabi_9d_check(nslots=nslots)["calibrated_dressed"]
    print(
        f"nslots={nslots:4d} F_process={row['fidelity']:.12f} "
        f"leakage_proxy={row['leakage_proxy']:.6e} theta={row['theta']:.9f}"
    )


nslots=  80 F_process=0.999979348002 leakage_proxy=3.184802e-06 theta=5.405639701
nslots= 160 F_process=0.999978594566 leakage_proxy=2.227614e-06 theta=5.405667509
nslots= 320 F_process=0.999978375514 leakage_proxy=2.063945e-06 theta=5.405674447
nslots= 640 F_process=0.999978318831 leakage_proxy=2.027758e-06 theta=5.405676180
nslots= 800 F_process=0.999978311977 leakage_proxy=2.023543e-06 theta=5.405676388
nslots=1000 F_process=0.999978307585 leakage_proxy=2.020859e-06 theta=5.405676521


## Conclusion

With the calibrated Eq. (2)-style model used in this notebook, the paper Eq. (1) pulse remains a high-fidelity phase solution even when the reported `237/303 MHz` single-photon Rabi split is retained.

The central calibrated-model ingredient is the two-photon resonance shift

```text
delta_res = (Omega_r^2 - Omega_b^2) / (4 Delta),
```

combined with the detuning-gauge waveform `delta(t) = -d phi/dt + delta_res`. Preparing and measuring in the bare computational branches already gives high process fidelity. Preparing and measuring in the leading-order dressed computational basis gives the retained best minimal-model value near `0.99998`.

This is still not a full reproduction of the experimental model. The full experiment includes multiple intermediate states, Clebsch-Gordan factors, the adjacent `m_J` Rydberg state, finite AOM rise/fall, scattering, Rydberg decay, measured dephasing/noise, finite-temperature blockade variation, and randomized-benchmark/SPAM extraction.


## Paper basis for the resonance-shift term

The paper does not introduce a symbol named `delta_res` or write the single-intermediate-state formula `(Omega_r^2 - Omega_b^2)/(4 Delta)`. That formula is the leading-order light-shift correction for the simplified Eq. (2) ladder used in this notebook.

The corresponding paper statements are:

- The blue 420-nm phase is represented as a time-dependent two-photon detuning in Eq. (2): `delta(t) proportional to -phi'(t)`; see Methods, "Dark states in two-photon Rydberg gates" in Evered et al., Nature 622, 268-272 (2023), lines 232-236. Source: https://www.nature.com/articles/s41586-023-06481-y
- The dark/bright-state basis is defined at two-photon resonance, and the effective energy splitting is set by AC Stark shifts; see lines 237-243 of the same Methods section.
- For the smooth-amplitude gate discussion, the authors state that they set the reference point so that equal 420/1013-nm Rabi rates are on two-photon resonance and have two-photon Rabi frequency `Omega`; see lines 215-221.
- Experimentally, the gate is calibrated by scanning global gate parameters and can absorb systematic offsets; see Fig. 1d / main text lines 108 and 112, and Extended Data Fig. 5 caption lines 798-800.

Thus, the notebook's `delta_res` should be read as the minimal single-intermediate-state implementation of the paper's calibrated two-photon resonance reference, not as a separately quoted parameter from the article.
